In [1]:
!rm .fleche -rf

# Getting Started with Fleche

This notebook demonstrates the main features of the `fleche` library, a caching library for Python.

## Long-running calculation

In [2]:
import time
from fleche import fleche, cache, tags, project
from fleche.digest import Digest

LOGGING INFO: local config fleche.toml does not exist, trying global
LOGGING INFO: global config /home/jules/.fleche.toml does not exist
LOGGING INFO: local config fleche.toml does not exist, trying global
LOGGING INFO: global config /home/jules/.fleche.toml does not exist


In [3]:
@fleche
def long_running_calculation(x):
    print(f'Running calculation for {x}...')
    time.sleep(2)
    return x * x

In [4]:
start = time.time()
long_running_calculation(2)
end = time.time()
print(f'First call took {end - start:.2f} seconds.')

Running calculation for 2...


First call took 2.00 seconds.


In [5]:
start = time.time()
long_running_calculation(2)
end = time.time()
print(f'Second call took {end - start:.2f} seconds.')

Second call took 0.00 seconds.


As you can see, the second call returns almost instantly, because the result was cached.

## Recursive function

In [6]:
@fleche
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

In [7]:
start = time.time()
fib(20)
end = time.time()
print(f'fib(20) took {end - start:.4f} seconds with caching.')

fib(20) took 0.0122 seconds with caching.


Without caching, this would be much slower as each call to `fib` would be recomputed.

## Caching Methods of User-defined Types

`fleche` can also cache methods of classes. For this to work, the class must be "digest-compatible". You can make a class digest-compatible by implementing a `__digest__` method or by using a `dataclass`.

In [8]:
class MyClass:
    def __init__(self, val):
        self.val = val
    
    def __digest__(self):
        # The digest defines how the instance is identified in the cache
        return Digest(str(self.val))

    @fleche
    def compute(self, x):
        print(f"Computing {self.val} + {x}...")
        time.sleep(1)
        return self.val + x

In [9]:
obj = MyClass(10)

start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"First call took {time.time() - start:.2f} seconds.")

start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"Second call (same instance) took {time.time() - start:.2f} seconds.")

Computing 10 + 5...


Result: 15
First call took 1.00 seconds.
Result: 15
Second call (same instance) took 0.00 seconds.


If you mutate the instance such that its digest changes, the cache will be missed.

In [10]:
obj.val = 20
start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"Call after mutation took {time.time() - start:.2f} seconds.")

Computing 20 + 5...


Result: 25
Call after mutation took 1.00 seconds.


## Passing Digests as Arguments

`fleche` supports passing `Digest` objects directly to cached functions. When a function receives a `Digest`, `fleche` automatically expands it to its actual value from the cache before executing the function. You can use the convenience wrapper `D` to mark a string as a digest.

In [11]:
from fleche import D

@fleche
def double(x):
    print(f"Doubling {x}...")
    return x * 2

# 1. Run the calculation to ensure it is cached
long_running_calculation(10)

# 2. Get its digest
digest_10 = long_running_calculation.digest(10)
print(f"Digest: {digest_10}")
# Output: 727c0fb9120f5eca7e0d4e550c6ee8d9019e07c62463b2a8cc7581d6cb8500

# 3. Pass the digest (even a short one!) to double(). It will be expanded to 100.
# Use D() to mark it as a digest.
print(f"Result: {double(D('727c0fb9'))}")

Running calculation for 10...


Digest: 727c0fb9120f5eca7e0d4e550c6ee8d9019e07c62463b2a8cc7581d8b9c700da
Doubling 100...
Result: 200


## Metadata

`fleche` allows you to add metadata to your cached functions using the `tags` context manager. This can be useful for organizing and querying your results.

In [12]:
@fleche
def another_calculation(a, b):
    return a + b

In [13]:
with tags(project='my_project', category='testing'):
    another_calculation(1, 2)
    another_calculation(3, 4)

This metadata is stored alongside the cached result. You can then use the `metadata.table` method to view the metadata for all cached results.

In [14]:
cache().table()

,name,args,kwargs,module,version,result,timestart,timestop,walltime,project,category
0,long_running_calculation,(92a9b214d814a4a7b5f9ba52e2248c6d16ec0196d8cd7...,{},__main__,None,83ada2198553b88cb3d0882f7fca8c4e9531049b978df3...,1.771246e+09,1.771246e+09,2.000444,NaN,NaN
1,fib,(da217d50752f3371d9f8b62a3e72409592bd34b74e14f...,{},__main__,None,da217d50752f3371d9f8b62a3e72409592bd34b74e14fd...,1.771246e+09,1.771246e+09,0.000013,NaN,NaN
2,fib,(26461de90bccabfde78889464de27c40b1a962fdc312a...,{},__main__,None,26461de90bccabfde78889464de27c40b1a962fdc312a7...,1.771246e+09,1.771246e+09,0.000015,NaN,NaN
3,fib,(92a9b214d814a4a7b5f9ba52e2248c6d16ec0196d8cd7...,{},__main__,None,da217d50752f3371d9f8b62a3e72409592bd34b74e14fd...,1.771246e+09,1.771246e+09,0.001268,NaN,NaN
4,fib,(65d52a82c5a72f12ca0499522dc9274a0e6822e103863...,{},__main__,None,92a9b214d814a4a7b5f9ba52e2248c6d16ec0196d8cd77...,1.771246e+09,1.771246e+09,0.002973,NaN,NaN
5,fib,(83ada2198553b88cb3d0882f7fca8c4e9531049b978df...,{},__main__,None,65d52a82c5a72f12ca0499522dc9274a0e6822e1038630...,1.771246e+09,1.771246e+09,0.003490,NaN,NaN
6,fib,(5b07a837d91d67764109c11fb912c9b91b4e9d9d4c909...,{},__main__,None,5b07a837d91d67764109c11fb912c9b91b4e9d9d4c909f...,1.771246e+09,1.771246e+09,0.004014,NaN,NaN
7,fib,(6511a915535857f7e7ca73b8b62968975f357664ed331...,{},__main__,None,08fa94f35017cb4c1e6d1a7edd09503f5f1650b888cbb6...,1.771246e+09,1.771246e+09,0.004507,NaN,NaN
8,fib,(307ea96b8b27c58a4c93e2e34e3783e475dc0e8ef99b4...,{},__main__,None,05c7cb7b6fa10504e50f57bdfb56962669740ee04b34ba...,1.771246e+09,1.771246e+09,0.004920,NaN,NaN
9,fib,(08fa94f35017cb4c1e6d1a7edd09503f5f1650b888cbb...,{},__main__,None,8d211f6913c5bc9a8f36627fdff1cf8ba87a64a9738ab9...,1.771246e+09,1.771246e+09,0.005397,NaN,NaN


## Filtering

The metadata table is just pandas so you can query and filter as you like.

In [15]:
cache().table().query('name!="fib"')

,name,args,kwargs,module,version,result,timestart,timestop,walltime,project,category
0,long_running_calculation,(92a9b214d814a4a7b5f9ba52e2248c6d16ec0196d8cd7...,{},__main__,None,83ada2198553b88cb3d0882f7fca8c4e9531049b978df3...,1.771246e+09,1.771246e+09,2.000444,NaN,NaN
22,compute,"(10, 5b07a837d91d67764109c11fb912c9b91b4e9d9d4...",{},__main__,None,4299b052f31ce284d363962dfed96eb3224cbafde29a40...,1.771246e+09,1.771246e+09,1.000408,NaN,NaN
23,compute,"(20, 5b07a837d91d67764109c11fb912c9b91b4e9d9d4...",{},__main__,None,7eabebe706259302bd54c2a51fd4350695d9481b4fbf09...,1.771246e+09,1.771246e+09,1.000372,NaN,NaN
24,long_running_calculation,(f3f64312ae6c7771851d6d1ab57d134ac193e54640a5c...,{},__main__,None,60079f7901a9295349d1796c037afc132e81286f785dde...,1.771246e+09,1.771246e+09,2.000346,NaN,NaN
25,double,"(727c0fb9,)",{},__main__,None,0c55ce113e97ef4f8899beb58e9aaffa3fffc56c6b4626...,1.771246e+09,1.771246e+09,0.000247,NaN,NaN
26,another_calculation,(da217d50752f3371d9f8b62a3e72409592bd34b74e14f...,{},__main__,None,65d52a82c5a72f12ca0499522dc9274a0e6822e1038630...,1.771246e+09,1.771246e+09,0.000033,my_project,testing
27,another_calculation,(65d52a82c5a72f12ca0499522dc9274a0e6822e103863...,{},__main__,None,307ea96b8b27c58a4c93e2e34e3783e475dc0e8ef99b4c...,1.771246e+09,1.771246e+09,0.000020,my_project,testing


In [16]:
cache().table().query('project=="my_project"')

,name,args,kwargs,module,version,result,timestart,timestop,walltime,project,category
26,another_calculation,(da217d50752f3371d9f8b62a3e72409592bd34b74e14f...,{},__main__,None,65d52a82c5a72f12ca0499522dc9274a0e6822e1038630...,1.771246e+09,1.771246e+09,0.000033,my_project,testing
27,another_calculation,(65d52a82c5a72f12ca0499522dc9274a0e6822e103863...,{},__main__,None,307ea96b8b27c58a4c93e2e34e3783e475dc0e8ef99b4c...,1.771246e+09,1.771246e+09,0.000020,my_project,testing
